In [1]:
#!/usr/bin/env python3
import numpy as np
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_algorithms.optimizers import SLSQP
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit.primitives import StatevectorEstimator

print(" FULL H₂ VQE — Electronic + Nuclear Repulsion")

# =====================================================
#  H₂ MOLECULE SETUP
# =====================================================
print("\n PySCF electronic structure...")
driver = PySCFDriver(atom='H 0 0 0; H 0 0 0.74', basis='sto3g')
esp = driver.run()

print(f"   Spatial orbitals: {esp.num_spatial_orbitals}")
print(f"   Particles: {esp.num_particles}")
print(f"   Nuclear repulsion: {esp.nuclear_repulsion_energy:.6f} Ha")

# =====================================================
# ELECTRONIC HAMILTONIAN → PAULI
# =====================================================
print("\n Jordan-Wigner transformation...")
mapper = JordanWignerMapper()
second_q_op = esp.hamiltonian.second_q_op()
qubit_op = mapper.map(second_q_op)
print(f"   Pauli terms: {len(qubit_op)}")
print(f"   Qubits: {qubit_op.num_qubits}")

# =====================================================
#  HARTRFOCK + UCCSD ANSATZ
# =====================================================
print("\n Hartree-Fock + UCCSD...")
hf = HartreeFock(esp.num_spatial_orbitals, esp.num_particles, mapper)
ansatz = UCCSD(esp.num_spatial_orbitals, esp.num_particles, mapper, initial_state=hf)
print(f"   Parameters: {ansatz.num_parameters}")

# =====================================================
# VQE OPTIMIZATION (ELECTRONIC ENERGY)
# =====================================================
print("\n SLSQP variational optimization...")
estimator = StatevectorEstimator()
optimizer = SLSQP(maxiter=200)

vqe = VQE(estimator, ansatz, optimizer)
vqe.initial_point = np.zeros(ansatz.num_parameters)

print("   Computing ⟨ψ|H_electronic|ψ⟩...")
result = vqe.compute_minimum_eigenvalue(qubit_op)

# =====================================================
#  TOTAL MOLECULAR ENERGY
# =====================================================
print("\n" + "="*70)
print(" H₂ GROUND STATE ENERGIES")
print("="*70)
electronic_energy = float(result.eigenvalue.real)
nuclear_repulsion = esp.nuclear_repulsion_energy
total_energy = electronic_energy + nuclear_repulsion

print(f" Electronic (VQE):     {electronic_energy:.6f} Ha")
print(f"  Nuclear repulsion:    {nuclear_repulsion:.6f} Ha")
print(f" TOTAL molecular:       {total_energy:.6f} Ha")
print(f" Expected total:        -1.137 Ha")
print(f" Error:                 {abs(total_energy + 1.137):.1e} Ha")

print(f"  Params optimized:     {len(result.optimal_parameters)}")
print(f"  Iterations:           {result.optimizer_result.nfev}")




 FULL H₂ VQE — Electronic + Nuclear Repulsion

 PySCF electronic structure...
   Spatial orbitals: 2
   Particles: (1, 1)
   Nuclear repulsion: 0.715104 Ha

 Jordan-Wigner transformation...
   Pauli terms: 15
   Qubits: 4

 Hartree-Fock + UCCSD...
   Parameters: 3

 SLSQP variational optimization...
   Computing ⟨ψ|H_electronic|ψ⟩...

 H₂ GROUND STATE ENERGIES
 Electronic (VQE):     -1.852388 Ha
  Nuclear repulsion:    0.715104 Ha
 TOTAL molecular:       -1.137284 Ha
 Expected total:        -1.137 Ha
 Error:                 2.8e-04 Ha
  Params optimized:     3
  Iterations:           10


/opt/miniconda3/envs/qiskit_gaara/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/miniconda3/envs/qiskit_gaara/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
